# Q1 — Comparison with Dynamic World and CGLS-LC100 (two seasons)

Reads the outputs of `01_landcover_classification.ipynb` (`03_classification_outputs.zip`) and compares the random-forest cell labels with
**Dynamic World V1** (the same 48 Sentinel-2 scenes, Oct–Dec 2019 + Feb–Mar 2019, probabilities averaged over all images pooled) and **CGLS-LC100 Collection 3, 2019**.

* **Edit only cell 2.**
* Outputs are added to the unzipped `q1_outputs/` folder and zipped as `05_final_outputs.zip`.
* Order: 1–8 → 8c–8f (extra agreement, diagnosis cells, DW time series, panels) → 8b (sensitivity, Oct–Dec only) → 9 (figure) → 10 (log and zip).
* Product-vs-product results are **agreement**, not accuracy. Visual-label comparisons use only 6 cells (3 High-confidence) and are reported as counts with 95% Wilson intervals.

In [ ]:
# 1. Install packages and import libraries
!pip -q install earthengine-api pyproj requests matplotlib rasterio scikit-learn==1.6.1 joblib shapely

import os, io, re, csv, json, math, shutil, zipfile, platform, datetime
import numpy as np
import requests
import ee
import pyproj
from pyproj import Transformer
import rasterio
from rasterio.features import rasterize
import sklearn
import matplotlib
import matplotlib.dates
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from PIL import Image

print("earthengine-api", ee.__version__, "| rasterio", rasterio.__version__,
      "| scikit-learn", sklearn.__version__, "| numpy", np.__version__)
if sklearn.__version__ != "1.6.1":
    print("WARNING: scikit-learn is not 1.6.1. Restart the runtime and run from cell 1 again.")

In [ ]:
# 2. Settings (the only cell to edit)
EE_PROJECT = "ee-sawargaonkaraniket"

# --- Inputs from 01_landcover_classification.ipynb
Q1_ZIP   = "03_classification_outputs.zip"   # uploaded if not already in the Colab folder
Q1_DIR   = "q1_outputs"                 # unzipped here
OUT_DIR  = Q1_DIR                       # comparison outputs are added to the same folder
ZIP_NAME = "05_final_outputs"   # final zip (cell 10)

# --- Windows (end date excluded) and masking: same as the composites
WINDOWS = {"octdec": ("2019-10-01", "2020-01-01"), "febmar": ("2019-02-01", "2019-04-01")}
CS_COLLECTION = "GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED"
CS_THRESHOLD  = 0.60

# --- Dynamic World
DW_COLLECTION       = "GOOGLE/DYNAMICWORLD/V1"
DW_MATCH_S2_SCENES  = True     # use only the Sentinel-2 scenes listed in s2_images_used*.csv (same 48 images)
DW_APPLY_CLOUDSCORE = False    # no extra Cloud Score+ mask on Dynamic World
DW_BANDS = ["water", "trees", "grass", "flooded_vegetation", "crops",
            "shrub_and_scrub", "built", "bare", "snow_and_ice"]          # DW label values 0-8
# Main run: mean of each class probability over ALL images of both windows pooled; highest class per
# pixel; valid = at least one observation. Sensitivity (cell 8b): mean over the Oct-Dec images only.

# --- CGLS-LC100 Collection 3, 2019
CGLS_ASSET      = "COPERNICUS/Landcover/100m/Proba-V-C3/Global/2019"
CGLS_CLASS_BAND = "discrete_classification"
# 10 m grid by nearest neighbour; no partial-pixel area weighting

# --- Classes: my six, then comparator-only classes
CLASS_NAMES = ["Shrubs", "Herbaceous vegetation", "Cropland", "Built-up",
               "Bare or sparse vegetation", "Permanent water"]
COMPARATOR_ONLY = ["Forest", "Herbaceous wetland", "Other"]     # the RF cannot predict these
ALL_CLASSES = CLASS_NAMES + COMPARATOR_ONLY

DW_CROSSWALK = {"water": "Permanent water", "trees": "Forest", "grass": "Herbaceous vegetation",
                "flooded_vegetation": "Herbaceous wetland", "crops": "Cropland",
                "shrub_and_scrub": "Shrubs", "built": "Built-up", "bare": "Bare or sparse vegetation",
                "snow_and_ice": "Other"}
CGLS_CROSSWALK = {20: "Shrubs", 30: "Herbaceous vegetation", 40: "Cropland", 50: "Built-up",
                  60: "Bare or sparse vegetation", 80: "Permanent water", 90: "Herbaceous wetland",
                  111: "Forest", 112: "Forest", 113: "Forest", 114: "Forest", 115: "Forest", 116: "Forest",
                  121: "Forest", 122: "Forest", 123: "Forest", 124: "Forest", 125: "Forest", 126: "Forest",
                  70: "Other", 100: "Other", 200: "Other"}                # 0 = no data

MIXED_THRESHOLD = 0.50     # one class only if its share is MORE than 50%, otherwise "Mixed (top and second)"
LOW_VALID_FLAG  = 0.50     # flag (not drop) a cell with a lower valid fraction

# --- My final visual labels (label, confidence)
VISUAL_LABELS = {
    "C01": ("Permanent water", "High"),
    "C02": ("Shrubs", "High"),
    "C05": ("Permanent water", "High"),
    "C08": ("Cropland", "Medium"),
    "C10": ("Mixed (Built-up and Shrubs)", "Medium"),
    "C11": ("Mixed (Cropland and Shrubs)", "Medium"),
}

# --- Diagnosis cells: the rule in cell 8d picks two cells; this override replaces them (None = use the rule)
DIAG_OVERRIDE        = ["C02", "C10"]
DIAG_OVERRIDE_REASON = ("C01 replaced by C02: C01 is permanent water (reservoir), so it was not considered "
                        "for diagnosis. Decided after the rule-based selection (C01, C10) had been seen.")

# --- Random forest for the sensitivity run (same settings as Q1)
RANDOM_SEED = 2019
RF_TREES    = 300

# --- Okabe-Ito colours (same as the Q1 class map), plus comparator-only classes
CLASS_COLOURS = {"Shrubs": "#D55E00", "Herbaceous vegetation": "#F0E442", "Cropland": "#E69F00",
                 "Built-up": "#555555", "Bare or sparse vegetation": "#CC79A7", "Permanent water": "#0072B2",
                 "Forest": "#009E73", "Herbaceous wetland": "#56B4E9", "Other": "#000000"}

In [ ]:
# 3. Earth Engine sign-in (interactive; no keys or tokens in the code)
ee.Authenticate()
ee.Initialize(project=EE_PROJECT)
print("Earth Engine initialised for project", EE_PROJECT)

In [ ]:
# 4. Load the Q1 outputs: grid, image lists, RF labels; shared helper functions
if not os.path.exists(Q1_ZIP) and not os.path.isdir(Q1_DIR):
    from google.colab import files
    print(f"{Q1_ZIP} not found: choose it to upload.")
    Q1_ZIP = list(files.upload())[0]
if not os.path.isdir(Q1_DIR):
    with zipfile.ZipFile(Q1_ZIP) as z:
        z.extractall(Q1_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

def q1(path):
    return os.path.join(Q1_DIR, path)

with open(q1("grid_cells.geojson")) as f:
    CELLS = [ft["properties"] for ft in json.load(f)["features"]]
with open(q1("aoi_boundary.geojson")) as f:
    AOI = json.load(f)["features"][0]["properties"]
with open(q1("run_log.json")) as f:
    Q1_LOG = json.load(f)
with open(q1("six_cells_for_labelling.json")) as f:
    SIX = json.load(f)

UTM_EPSG = AOI["utm_epsg"]; EPSG = f"EPSG:{UTM_EPSG}"
X0, Y0, X1, Y1 = AOI["xmin"], AOI["ymin"], AOI["xmax"], AOI["ymax"]
PIXEL_M = 10
CELL_PX = int((CELLS[0]["xmax"] - CELLS[0]["xmin"]) / PIXEL_M)        # 50
GRID_W, GRID_H = int((X1 - X0) / PIXEL_M), int((Y1 - Y0) / PIXEL_M)   # 200 x 150
CELL_IDS = [c["cell_id"] for c in CELLS]
to_utm = Transformer.from_crs("EPSG:4326", EPSG, always_xy=True)
to_wgs = Transformer.from_crs(EPSG, "EPSG:4326", always_xy=True)

def read_csv(path):
    with open(path, newline="") as f:
        return list(csv.DictReader(f))
def write_csv(path, rows, fields=None):
    fields = fields or list(rows[0].keys())
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        w.writerows(rows)

S2_IDS = {"octdec": [r["image_id"] for r in read_csv(q1("s2_images_used.csv"))],
          "febmar": [r["image_id"] for r in read_csv(q1("s2_images_used_febmar.csv"))]}
RF_ROWS = {r["cell_id"]: r for r in read_csv(q1("rf_cell_labels.csv"))}

# --- shared helpers
def cell_window(c):
    """Row/column slices of a cell inside the 200 x 150 grid arrays."""
    r0, c0 = int((Y1 - c["ymax"]) / PIXEL_M), int((c["xmin"] - X0) / PIXEL_M)
    return slice(r0, r0 + CELL_PX), slice(c0, c0 + CELL_PX)

def label_from_shares(shares):
    """shares: dict class -> fraction of valid pixels. Returns (label, top, top_share, second, second_share)."""
    if not shares or sum(shares.values()) == 0:
        return "No valid data", "", "", "", ""
    order = sorted(shares, key=lambda k: (-shares[k], ALL_CLASSES.index(k)))
    top, sec = order[0], order[1]
    label = top if shares[top] > MIXED_THRESHOLD else f"Mixed ({top} and {sec})"
    return label, top, round(shares[top], 4), sec, round(shares[sec], 4)

def class_shares(class_map, valid):
    """class_map: int array of ALL_CLASSES indices. Returns dict class -> share of valid pixels."""
    n = int(valid.sum())
    if n == 0:
        return {}
    vals = class_map[valid]
    return {k: float((vals == i).sum()) / n for i, k in enumerate(ALL_CLASSES)}

MIXED_RE = re.compile(r"^Mixed \((.+) and (.+)\)$")
def label_set(label):
    """'Shrubs' -> {'Shrubs'}; 'Mixed (A and B)' -> {'A', 'B'}; 'No valid data' or '' -> empty set."""
    if not label or label == "No valid data":
        return set()
    m = MIXED_RE.match(label)
    return {m.group(1), m.group(2)} if m else {label}

def same_label(a, b):
    """Strict: identical single class, or both Mixed with the same two classes."""
    sa, sb = label_set(a), label_set(b)
    return bool(sa) and sa == sb and (len(sa) == 1) == (len(sb) == 1)

def ee_rect(bounds):
    return ee.Geometry.Rectangle(list(bounds), proj=EPSG, geodesic=False)

def download_geotiff(img, path, band_names, bounds=(X0, Y0, X1, Y1)):
    """Download img on the 10 m grid covering bounds; masked pixels stay masked."""
    w, h = int((bounds[2] - bounds[0]) / PIXEL_M), int((bounds[3] - bounds[1]) / PIXEL_M)
    url = img.select(band_names).getDownloadURL({
        "crs": EPSG, "crs_transform": [PIXEL_M, 0, bounds[0], 0, -PIXEL_M, bounds[3]],
        "dimensions": f"{w}x{h}", "format": "GEO_TIFF", "filePerBand": False})
    r = requests.get(url, timeout=600)
    r.raise_for_status()
    content = r.content
    if content[:2] == b"PK":
        with zipfile.ZipFile(io.BytesIO(content)) as z:
            content = z.read([n for n in z.namelist() if n.endswith(".tif")][0])
    with open(path, "wb") as f:
        f.write(content)
    with rasterio.open(path, "r+") as dst:
        dst.descriptions = tuple(band_names)
    with rasterio.open(path) as src:
        t = src.transform
        ok = (src.width == w and src.height == h and src.crs.to_epsg() == UTM_EPSG
              and abs(t.c - bounds[0]) < 1e-6 and abs(t.f - bounds[3]) < 1e-6)
    print(f"{os.path.basename(path)}: {w}x{h} px, {len(band_names)} bands -> "
          f"{'grid check OK' if ok else 'WARNING: grid does not match'}")

def read_bands(path):
    """(bands, H, W) float32 with NaN where masked or nodata."""
    with rasterio.open(path) as src:
        return src.read(masked=True).astype("float32").filled(np.nan), src.transform

def write_class_map(path, class_map, transform):
    with rasterio.open(path, "w", driver="GTiff", width=GRID_W, height=GRID_H, count=1, dtype="uint8",
                       nodata=255, crs=EPSG, transform=transform) as dst:
        dst.write(class_map.astype("uint8"), 1)
        dst.update_tags(classes=json.dumps({i: n for i, n in enumerate(ALL_CLASSES)}))

print(f"Grid {EPSG} x {X0}-{X1}, y {Y0}-{Y1} ({GRID_W} x {GRID_H} px), {len(CELLS)} cells")
print(f"S2 scenes: {len(S2_IDS['octdec'])} Oct-Dec, {len(S2_IDS['febmar'])} Feb-Mar | RF labels for {len(RF_ROWS)} cells")
if set(VISUAL_LABELS) != set(SIX["cells"]):
    print("WARNING: VISUAL_LABELS cells differ from six_cells_for_labelling.json", SIX["cells"])

In [ ]:
# 5. Dynamic World on the grid: mean probability over ALL images of both windows pooled
CS = ee.ImageCollection(CS_COLLECTION)
AOI_EE = ee_rect((X0, Y0, X1, Y1))
DW_COLS, DW_IMAGE_INFO = {}, {}
for name, (start, end) in WINDOWS.items():
    col = ee.ImageCollection(DW_COLLECTION).filterBounds(AOI_EE).filterDate(start, end)
    available = col.aggregate_array("system:index").getInfo()
    if DW_MATCH_S2_SCENES:
        col = col.filter(ee.Filter.inList("system:index", S2_IDS[name]))
    used = col.aggregate_array("system:index").getInfo()
    if not used:
        raise RuntimeError(f"No Dynamic World images match the {name} Sentinel-2 scenes. "
                           "Set DW_MATCH_S2_SCENES = False in cell 2 and record that choice.")
    if DW_APPLY_CLOUDSCORE:
        col = col.linkCollection(CS, ["cs_cdf"]).map(
            lambda img: img.updateMask(img.select("cs_cdf").gte(CS_THRESHOLD)))
    DW_COLS[name] = col
    DW_IMAGE_INFO[name] = {"available": sorted(available), "used": sorted(used),
                           "s2_scenes_without_dw": sorted(set(S2_IDS[name]) - set(used))}
    print(f"Dynamic World {name}: {len(available)} images available, {len(used)} used "
          f"(of {len(S2_IDS[name])} Sentinel-2 scenes)")
DW_N_IMAGES = sum(len(DW_IMAGE_INFO[w]["used"]) for w in WINDOWS)

def dw_mean_image(col):
    """Mean of each class probability over all images, number of observations, most frequent label."""
    return ee.Image.cat([col.select(DW_BANDS).mean(),
                         col.select("label").count().unmask(0).rename("n_obs"),
                         col.select("label").mode().rename("label_mode")]).toFloat()

DW_TIF_BANDS = DW_BANDS + ["n_obs", "label_mode"]
DW_POOLED = DW_COLS["octdec"].merge(DW_COLS["febmar"])
DW_TIF = os.path.join(OUT_DIR, "dw_mean_probabilities.tif")
download_geotiff(dw_mean_image(DW_POOLED), DW_TIF, DW_TIF_BANDS)

# --- local processing
DW_TO_IDX = np.array([ALL_CLASSES.index(DW_CROSSWALK[b]) for b in DW_BANDS])

def load_dw(path):
    arr, transform = read_bands(path)
    probs, nobs, mode_raw = arr[:len(DW_BANDS)], np.nan_to_num(arr[len(DW_BANDS)], nan=0), arr[len(DW_BANDS) + 1]
    valid = (nobs > 0) & np.isfinite(probs).all(axis=0)            # at least one observation
    return probs, nobs, mode_raw, valid, transform

def dw_products(probs, valid):
    """Highest DW class per valid pixel, crosswalked to ALL_CLASSES indices (255 = no data)."""
    cmap = np.full((GRID_H, GRID_W), 255, dtype="uint8")
    if valid.any():
        cmap[valid] = DW_TO_IDX[np.nanargmax(probs[:, valid], axis=0)]
    return cmap

def mode_products(mode_raw):
    ok = np.isfinite(mode_raw)
    cmap = np.full((GRID_H, GRID_W), 255, dtype="uint8")
    cmap[ok] = DW_TO_IDX[mode_raw[ok].astype(int)]
    return cmap, ok

def dw_cell_rows(cmap, valid, probs, nobs, n_images, mode_map=None, mode_valid=None):
    rows = {}
    for c in CELLS:
        rs, cs = cell_window(c)
        v = valid[rs, cs]
        shares = class_shares(cmap[rs, cs], v)
        label, top, ts, sec, ss = label_from_shares(shares)
        nv = int(v.sum())
        pc = probs[:, rs, cs][:, v]
        row = {"cell_id": c["cell_id"], "dw_label": label, "dw_top_class": top, "dw_top_share": ts,
               "dw_second_class": sec, "dw_second_share": ss,
               "pixels_in_cell": CELL_PX * CELL_PX, "valid_pixels": nv,
               "valid_fraction": round(nv / CELL_PX ** 2, 4),
               "low_valid_flag": nv / CELL_PX ** 2 < LOW_VALID_FLAG,
               "mean_obs": round(float(nobs[rs, cs].mean()), 3), "n_images": n_images,
               "dw_mean_pixel_confidence": round(float(pc.max(axis=0).mean()), 4) if nv else ""}
        row.update({f"share_{k}": round(shares.get(k, 0.0), 4) if nv else "" for k in ALL_CLASSES})
        row.update({f"meanprob_{b}": round(float(pc[i].mean()), 4) if nv else "" for i, b in enumerate(DW_BANDS)})
        if mode_map is not None:
            row["dw_mode_label"] = label_from_shares(class_shares(mode_map[rs, cs], mode_valid[rs, cs]))[0]
        rows[c["cell_id"]] = row
    return rows

P_MAIN, NOBS_MAIN, MODE_RAW, DW_VALID, GRID_TRANSFORM = load_dw(DW_TIF)
DW_MAP = dw_products(P_MAIN, DW_VALID)
DW_MODE_MAP, MODE_VALID = mode_products(MODE_RAW)
write_class_map(os.path.join(OUT_DIR, "dw_class_map.tif"), DW_MAP, GRID_TRANSFORM)

DW_ROWS = dw_cell_rows(DW_MAP, DW_VALID, P_MAIN, NOBS_MAIN, DW_N_IMAGES, DW_MODE_MAP, MODE_VALID)
write_csv(os.path.join(OUT_DIR, "dw_cell_shares.csv"), list(DW_ROWS.values()))
write_csv(os.path.join(OUT_DIR, "dw_images_used.csv"),
          [{"window": w, "image_id": i, "used": i in DW_IMAGE_INFO[w]["used"]}
           for w in WINDOWS for i in DW_IMAGE_INFO[w]["available"]])
for r in DW_ROWS.values():
    share = f"{r['dw_top_share']:.1%}" if r["dw_top_share"] != "" else ""
    print(f"{r['cell_id']}  DW {r['dw_label']:<50} top {share:>6}  obs {r['mean_obs']:>5}  mode: {r['dw_mode_label']}")

In [ ]:
# 6. CGLS-LC100 2019 on the 10 m grid (nearest neighbour, no area weighting)
cgls = ee.Image(CGLS_ASSET)
cgls_band_names = cgls.bandNames().getInfo()
CGLS_PROBA_BAND = next((b for b in cgls_band_names if "proba" in b), None)
cgls_proj = cgls.select(CGLS_CLASS_BAND).projection().getInfo()
CGLS_BANDS = [CGLS_CLASS_BAND] + ([CGLS_PROBA_BAND] if CGLS_PROBA_BAND else [])
CGLS_TIF = os.path.join(OUT_DIR, "cgls_2019_on_10m_grid.tif")
download_geotiff(cgls.select(CGLS_BANDS).toFloat(), CGLS_TIF, CGLS_BANDS)
print("CGLS native projection:", cgls_proj["crs"], cgls_proj.get("transform"),
      "| probability band:", CGLS_PROBA_BAND)

# --- local processing
cg_arr, _ = read_bands(CGLS_TIF)
CG_CODE = cg_arr[0]
CG_PROBA = cg_arr[1] if CGLS_PROBA_BAND else np.full_like(CG_CODE, np.nan)
CG_VALID = np.isfinite(CG_CODE) & (np.nan_to_num(CG_CODE) > 0)
CG_MAP = np.full((GRID_H, GRID_W), 255, dtype="uint8")
unknown_codes = set()
for code_ in np.unique(CG_CODE[CG_VALID]).astype(int):
    cls = CGLS_CROSSWALK.get(code_)
    if cls is None:
        unknown_codes.add(code_); cls = "Other"
    CG_MAP[CG_VALID & (CG_CODE == code_)] = ALL_CLASSES.index(cls)
if unknown_codes:
    print("WARNING: CGLS codes not in the crosswalk, counted as Other:", sorted(unknown_codes))
write_class_map(os.path.join(OUT_DIR, "cgls_class_map.tif"), CG_MAP, GRID_TRANSFORM)

# Native CGLS pixel of every 10 m pixel centre, computed in the CGLS projection's own CRS
a, _, c_, _, e, f_ = cgls_proj["transform"][:6]
to_cgls = Transformer.from_crs(EPSG, cgls_proj["crs"], always_xy=True)
cx = X0 + (np.arange(GRID_W) + 0.5) * PIXEL_M
cy = Y1 - (np.arange(GRID_H) + 0.5) * PIXEL_M
CX, CY = np.meshgrid(cx, cy)
nx, ny = to_cgls.transform(CX, CY)
NATIVE_COL = np.floor((nx - c_) / a).astype(np.int64)
NATIVE_ROW = np.floor((ny - f_) / e).astype(np.int64)
from_cgls = Transformer.from_crs(cgls_proj["crs"], EPSG, always_xy=True)

def native_pixel_counts(c, valid_cell):
    """(native pixels whose centre is inside the cell, native pixels touching the cell)."""
    rs, cs = cell_window(c)
    pairs = set(zip(NATIVE_COL[rs, cs][valid_cell].tolist(), NATIVE_ROW[rs, cs][valid_cell].tolist()))
    centred = 0
    for col_, row_ in pairs:
        ux, uy = from_cgls.transform(c_ + (col_ + 0.5) * a, f_ + (row_ + 0.5) * e)
        centred += (c["xmin"] <= ux < c["xmax"]) and (c["ymin"] <= uy < c["ymax"])
    return centred, len(pairs)

CGLS_ROWS = {}
for c in CELLS:
    rs, cs = cell_window(c)
    v = CG_VALID[rs, cs]
    shares = class_shares(CG_MAP[rs, cs], v)
    label, top, ts, sec, ss = label_from_shares(shares)
    nv = int(v.sum())
    codes = CG_CODE[rs, cs][v].astype(int)
    n_centred, n_touching = native_pixel_counts(c, v)
    row = {"cell_id": c["cell_id"], "cgls_label": label, "cgls_top_class": top, "cgls_top_share": ts,
           "cgls_second_class": sec, "cgls_second_share": ss,
           "pixels_in_cell": CELL_PX * CELL_PX, "valid_pixels": nv, "valid_fraction": round(nv / CELL_PX ** 2, 4),
           "low_valid_flag": nv / CELL_PX ** 2 < LOW_VALID_FLAG,
           "cgls_native_pixels": n_centred,               # native pixels whose centre is inside the cell
           "cgls_native_pixels_touching": n_touching,     # native pixels contributing any 10 m pixel
           "cgls_mean_proba": round(float(np.nanmean(CG_PROBA[rs, cs][v])), 2) if nv and CGLS_PROBA_BAND else "",
           "n_images": "n/a", "mean_obs": "n/a",
           "cgls_codes": " ".join(f"{k}:{int((codes == k).sum())}" for k in sorted(set(codes)))}
    row.update({f"share_{k}": round(shares.get(k, 0.0), 4) if nv else "" for k in ALL_CLASSES})
    CGLS_ROWS[c["cell_id"]] = row
write_csv(os.path.join(OUT_DIR, "cgls_cell_shares.csv"), list(CGLS_ROWS.values()))
for r in CGLS_ROWS.values():
    print(f"{r['cell_id']}  CGLS {r['cgls_label']:<45} native px {r['cgls_native_pixels']:>3} centred, "
          f"{r['cgls_native_pixels_touching']:>3} touching  codes {r['cgls_codes']}")

In [ ]:
# 7. Cell-level comparison table (strict same_label rule)
PAIRS = [("rf", "dw"), ("rf", "cgls"), ("dw", "cgls")]
VIS_PAIRS = [("visual", "rf"), ("visual", "dw"), ("visual", "cgls")]
SRC = {"rf": RF_ROWS, "dw": DW_ROWS, "cgls": CGLS_ROWS}

def top_of(label):
    """Top class of a label: the class itself, or the first class of 'Mixed (top and second)'."""
    m = MIXED_RE.match(label or "")
    return m.group(1) if m else (label if label and label != "No valid data" else "")

COMP_ROWS = []
for c in CELLS:
    cid = c["cell_id"]
    vis = VISUAL_LABELS.get(cid)
    row = {"cell_id": cid, "row": c["row"], "col": c["col"], "in_visual_sample": vis is not None,
           "visual_label": vis[0] if vis else "", "visual_confidence": vis[1] if vis else "",
           "pixels_in_cell": CELL_PX * CELL_PX}
    for p in ("rf", "dw", "cgls"):
        s = SRC[p][cid]
        sec_share = s[f"{p}_second_share"]
        no_second = sec_share in ("", None) or float(sec_share) == 0
        row.update({f"{p}_label": s[f"{p}_label"], f"{p}_top_class": s[f"{p}_top_class"],
                    f"{p}_top_share": s[f"{p}_top_share"],
                    f"{p}_second_class": "" if no_second else s[f"{p}_second_class"],   # blank when share is 0
                    f"{p}_second_share": sec_share, f"{p}_valid_pixels": s["valid_pixels"]})
    row.update({"dw_mean_pixel_confidence": DW_ROWS[cid]["dw_mean_pixel_confidence"],
                "dw_mean_obs": DW_ROWS[cid]["mean_obs"],
                "cgls_native_pixels": CGLS_ROWS[cid]["cgls_native_pixels"],
                "cgls_mean_proba": CGLS_ROWS[cid]["cgls_mean_proba"]})
    for a_, b_ in PAIRS:
        row[f"{a_}_vs_{b_}"] = same_label(row[f"{a_}_label"], row[f"{b_}_label"])
    for a_, b_ in VIS_PAIRS:
        row[f"{a_}_vs_{b_}"] = same_label(row["visual_label"], row[f"{b_}_label"]) if vis else ""
    distinct = {frozenset(label_set(row[f"{p}_label"])) for p in ("rf", "dw", "cgls")}
    row["distinct_method_labels"] = len(distinct)
    # --- extra columns (new)
    row.update({"rf_mean_pixel_confidence": RF_ROWS[cid]["rf_mean_pixel_confidence"],
                "dw_valid_fraction": DW_ROWS[cid]["valid_fraction"],
                "cgls_valid_fraction": CGLS_ROWS[cid]["valid_fraction"],
                "dw_mode_label": DW_ROWS[cid]["dw_mode_label"]})
    for p in ("dw", "cgls"):
        row[f"{p}_out_of_scheme"] = bool(label_set(row[f"{p}_label"]) & set(COMPARATOR_ONLY))
    COMP_ROWS.append(row)
write_csv(os.path.join(OUT_DIR, "comparison_cell_labels.csv"), COMP_ROWS)

for r in COMP_ROWS:
    v = f" | visual {r['visual_label']} ({r['visual_confidence']})" if r["in_visual_sample"] else ""
    print(f"{r['cell_id']}  RF {r['rf_label']:<42} DW {r['dw_label']:<42} CGLS {r['cgls_label']}{v}")

In [ ]:
# 8. Agreement summary: same label and same top class (counts with denominators)
SUMMARY = []
vis_rows = [r for r in COMP_ROWS if r["in_visual_sample"]]
def same_top(r, a_, b_):
    """Product pairs: identical top classes. Visual pairs: the product's top class is ANY class in the
    visual label (either class of a Mixed label)."""
    if a_ == "visual":
        return bool(r[f"{b_}_top_class"]) and r[f"{b_}_top_class"] in label_set(r["visual_label"])
    return bool(r[f"{a_}_top_class"]) and r[f"{a_}_top_class"] == r[f"{b_}_top_class"]

for a_, b_ in PAIRS + VIS_PAIRS:
    rows = vis_rows if a_ == "visual" else COMP_ROWS
    SUMMARY.append({
        "pair": f"{'Visual' if a_ == 'visual' else a_.upper()} vs {b_.upper()}",
        "cells_compared": len(rows),
        "same_label": sum(bool(r[f"{a_}_vs_{b_}"]) for r in rows),
        "same_top_class": sum(same_top(r, a_, b_) for r in rows)})
write_csv(os.path.join(OUT_DIR, "comparison_agreement_summary.csv"), SUMMARY)
for s in SUMMARY:
    print(f"{s['pair']:<16} same label {s['same_label']}/{s['cells_compared']}   "
          f"same top class {s['same_top_class']}/{s['cells_compared']}")

In [ ]:
# 8c. Extra agreement measures: lenient, share overlap, 95% Wilson intervals, High-only subset
def lenient(a, b):
    """A Mixed label matches either of its two classes; two Mixed labels match if they share a class."""
    sa, sb = label_set(a), label_set(b)
    if not sa or not sb:
        return False
    if len(sa) == 1 and len(sb) == 1:
        return sa == sb
    return bool(sa & sb)

def wilson(k, n, z=1.96):
    if n == 0:
        return "", ""
    p = k / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return round(max(0.0, centre - half), 3), round(min(1.0, centre + half), 3)

def shares_of(product, cid):
    """Class shares (fractions) for a cell; a class missing from a product counts as 0."""
    src = {"rf": RF_ROWS, "dw": DW_ROWS, "cgls": CGLS_ROWS}[product][cid]
    out = {}
    for k in ALL_CLASSES:
        v = src.get(f"share_{k}", "")
        out[k] = float(v) if v not in ("", None) else 0.0
    return out

def share_overlap(a, b, cid):
    sa, sb = shares_of(a, cid), shares_of(b, cid)
    return sum(min(sa[k], sb[k]) for k in ALL_CLASSES)

for r in COMP_ROWS:
    cid = r["cell_id"]
    for a_, b_ in PAIRS:
        r[f"{a_}_vs_{b_}_lenient"] = lenient(r[f"{a_}_label"], r[f"{b_}_label"])
        r[f"{a_}_vs_{b_}_share_overlap"] = round(share_overlap(a_, b_, cid), 4)
    for a_, b_ in VIS_PAIRS:
        r[f"{a_}_vs_{b_}_lenient"] = lenient(r["visual_label"], r[f"{b_}_label"]) if r["in_visual_sample"] else ""
    r["mean_share_overlap"] = round(np.mean([r[f"{a_}_vs_{b_}_share_overlap"] for a_, b_ in PAIRS]), 4)
    r["rf_frac_lowconf"] = RF_ROWS[cid].get("rf_frac_lowconf", "")
write_csv(os.path.join(OUT_DIR, "comparison_cell_labels.csv"), COMP_ROWS)

EXT_SUMMARY = []
def add_summary(pair_name, rows, a_, b_, with_overlap):
    n = len(rows)
    ks = sum(bool(r[f"{a_}_vs_{b_}"]) for r in rows)
    kl = sum(bool(r[f"{a_}_vs_{b_}_lenient"]) for r in rows)
    sl, sh = wilson(ks, n)
    ll, lh = wilson(kl, n)
    EXT_SUMMARY.append({
        "pair": pair_name, "subset": "all" if rows is not High_rows else "High only", "n": n,
        "strict": ks, "strict_wilson_low": sl, "strict_wilson_high": sh,
        "lenient": kl, "lenient_wilson_low": ll, "lenient_wilson_high": lh,
        "mean_share_overlap": round(float(np.mean([r[f"{a_}_vs_{b_}_share_overlap"] for r in rows])), 3)
                              if with_overlap and n else ""})

High_rows = [r for r in COMP_ROWS if r["in_visual_sample"] and r["visual_confidence"] == "High"]
for a_, b_ in PAIRS:
    add_summary(f"{a_.upper()} vs {b_.upper()}", COMP_ROWS, a_, b_, True)
for a_, b_ in VIS_PAIRS:
    add_summary(f"Visual vs {b_.upper()}", vis_rows, a_, b_, False)
for a_, b_ in VIS_PAIRS:
    add_summary(f"Visual vs {b_.upper()}", High_rows, a_, b_, False)
write_csv(os.path.join(OUT_DIR, "agreement_summary_extended.csv"), EXT_SUMMARY)

print(f"{'pair':<15}{'subset':<11}{'n':>3}   {'strict (95% CI)':<22}{'lenient (95% CI)':<22}mean overlap")
for s in EXT_SUMMARY:
    st = f"{s['strict']}/{s['n']} ({s['strict_wilson_low']}-{s['strict_wilson_high']})"
    le = f"{s['lenient']}/{s['n']} ({s['lenient_wilson_low']}-{s['lenient_wilson_high']})"
    print(f"{s['pair']:<15}{s['subset']:<11}{s['n']:>3}   {st:<22}{le:<22}{s['mean_share_overlap']}")

In [ ]:
# 8d. Diagnosis cells (rule fixed in advance; main run only)
DIAG_RULE = ("The two cells with the lowest mean share overlap across RF-DW, RF-CGLS and DW-CGLS "
             "(main two-season run only); ties broken by the lower rf_mean_pixel_confidence.")
ranked = sorted(COMP_ROWS, key=lambda r: (r["mean_share_overlap"], float(r["rf_mean_pixel_confidence"])))
DIAG_RULE_CELLS = [r["cell_id"] for r in ranked[:2]]
DIAG_CELLS = list(DIAG_OVERRIDE) if DIAG_OVERRIDE else DIAG_RULE_CELLS     # cells used in 8e, 8f and 8b
diag = {
    "rule": DIAG_RULE,
    "mean_share_overlap": {r["cell_id"]: r["mean_share_overlap"] for r in COMP_ROWS},
    "pair_overlaps": {r["cell_id"]: {f"{a_}_vs_{b_}": r[f"{a_}_vs_{b_}_share_overlap"] for a_, b_ in PAIRS}
                      for r in COMP_ROWS},
    "rf_mean_pixel_confidence": {r["cell_id"]: float(r["rf_mean_pixel_confidence"]) for r in COMP_ROWS},
    "rule_selected_cells": DIAG_RULE_CELLS,
    "override_cells": list(DIAG_OVERRIDE) if DIAG_OVERRIDE else None,
    "override_reason": DIAG_OVERRIDE_REASON if DIAG_OVERRIDE else None,
    "override_made_after_seeing_results": bool(DIAG_OVERRIDE),
    "diagnosis_cells": DIAG_CELLS,
}
with open(os.path.join(OUT_DIR, "diagnosis_cells.json"), "w") as f:
    json.dump(diag, f, indent=1)
for r in ranked:
    mark = ("  <- rule" if r["cell_id"] in DIAG_RULE_CELLS else "") + \
           ("  <- used for diagnosis" if r["cell_id"] in DIAG_CELLS else "")
    print(f"{r['cell_id']}  mean overlap {r['mean_share_overlap']:.3f}  "
          f"rf confidence {float(r['rf_mean_pixel_confidence']):.3f}{mark}")
if DIAG_OVERRIDE:
    print(f"\nOverride: rule selected {DIAG_RULE_CELLS}, using {DIAG_CELLS}. Reason: {DIAG_OVERRIDE_REASON}")

In [ ]:
# 8e. Dynamic World per-date time series for the two diagnosis cells (same collections as cell 5)
TS_RAW = {}
for cid in DIAG_CELLS:
    c = next(x for x in CELLS if x["cell_id"] == cid)
    geom = ee_rect((c["xmin"], c["ymin"], c["xmax"], c["ymax"]))
    rows = []
    for name in WINDOWS:
        def per_image(img, geom=geom, name=name):
            means = img.select(DW_BANDS).reduceRegion(ee.Reducer.mean(), geom, scale=PIXEL_M, crs=EPSG)
            count = img.select("label").reduceRegion(ee.Reducer.count(), geom, scale=PIXEL_M, crs=EPSG)
            return ee.Feature(None, means).set({"image_id": img.get("system:index"),
                                                "time": img.get("system:time_start"),
                                                "valid_pixels": count.get("label"), "window": name})
        fc = DW_COLS[name].map(per_image).getInfo()
        rows += [ft["properties"] for ft in fc["features"]]
    TS_RAW[cid] = rows

# --- write
TS = {}
for cid, rows in TS_RAW.items():
    out = []
    for p in sorted(rows, key=lambda r: r["time"]):
        rec = {"date": datetime.datetime.fromtimestamp(p["time"] / 1000, datetime.timezone.utc).strftime("%Y-%m-%d"),
               "image_id": p["image_id"], "window": p["window"], "valid_pixels": int(p.get("valid_pixels") or 0)}
        for k in ALL_CLASSES:     # crosswalked: sum of the DW bands that map to each class
            vals = [p.get(b) for b in DW_BANDS if DW_CROSSWALK[b] == k]
            rec[f"prob_{k}"] = round(sum(vals), 4) if vals and None not in vals else ""
        out.append(rec)
    TS[cid] = out
    write_csv(os.path.join(OUT_DIR, f"dw_timeseries_{cid}.csv"), out,
              ["date", "image_id", "window", "valid_pixels"] + [f"prob_{k}" for k in ALL_CLASSES])
    print(f"{cid}: {len(out)} Dynamic World dates "
          f"({sum(r['valid_pixels'] > 0 for r in out)} with valid pixels) -> dw_timeseries_{cid}.csv")

In [ ]:
# 8f. One diagnosis panel per cell: chips, class maps, share bars, DW time series
with rasterio.open(q1("rf_class_map.tif")) as src:
    RF_MAP6 = src.read(1)
RF_MAP = np.where(RF_MAP6 == 255, 255, RF_MAP6).astype("uint8")   # RF indices 0-5 = first six of ALL_CLASSES
CMAP_ALL = ListedColormap([CLASS_COLOURS[k] for k in ALL_CLASSES])

import textwrap

def wrap(text, width=28):
    return "\n".join(textwrap.wrap(text, width))

def show_class_map(ax, cmap_arr, title):
    ax.imshow(np.ma.masked_equal(cmap_arr, 255), cmap=CMAP_ALL, vmin=-0.5, vmax=len(ALL_CLASSES) - 0.5,
              interpolation="nearest")
    ax.set_title(wrap(title), fontsize=9); ax.set_xticks([]); ax.set_yticks([])

DIAG_PANELS = []
for cid in DIAG_CELLS:
    c = next(x for x in CELLS if x["cell_id"] == cid)
    rs, cs = cell_window(c)
    fig = plt.figure(figsize=(17, 9.5))
    gs = fig.add_gridspec(2, 5, height_ratios=[1, 1.25], hspace=0.28, wspace=0.32)
    for j, (folder, title) in enumerate([("chips", "Oct-Dec 2019 (B4/B3/B2, 0-3000)"),
                                          ("chips_febmar", "Feb-Mar 2019 (B4/B3/B2, 0-3000)")]):
        ax = fig.add_subplot(gs[0, j])
        ax.imshow(Image.open(q1(f"{folder}/{cid}.png")).convert("RGB"))
        ax.set_title(wrap(title), fontsize=9); ax.set_xticks([]); ax.set_yticks([])
    show_class_map(fig.add_subplot(gs[0, 2]), RF_MAP[rs, cs], f"RF: {RF_ROWS[cid]['rf_label']}")
    show_class_map(fig.add_subplot(gs[0, 3]), DW_MAP[rs, cs], f"Dynamic World: {DW_ROWS[cid]['dw_label']}")
    show_class_map(fig.add_subplot(gs[0, 4]), CG_MAP[rs, cs],
                   f"CGLS 2019: {CGLS_ROWS[cid]['cgls_label']} ({CGLS_ROWS[cid]['cgls_native_pixels']} native CGLS px)")

    # Share bars
    ax = fig.add_subplot(gs[1, 0:2])
    present = [k for k in ALL_CLASSES if any(shares_of(p, cid)[k] > 0 for p in ("rf", "dw", "cgls"))]
    x = np.arange(len(present)); wbar = 0.27
    for j, (p, hatch) in enumerate([("rf", ""), ("dw", "//"), ("cgls", "..")]):
        s = shares_of(p, cid)
        ax.bar(x + (j - 1) * wbar, [s[k] for k in present], wbar, label=p.upper(),
               color=[CLASS_COLOURS[k] for k in present], edgecolor="black", hatch=hatch, linewidth=0.6)
    ax.axhline(MIXED_THRESHOLD, color="grey", ls="--", lw=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(present, fontsize=8, rotation=35, ha="right", rotation_mode="anchor")
    ax.set_ylabel("Share of valid pixels"); ax.set_ylim(0, 1)
    ax.legend(handles=[Patch(facecolor="white", edgecolor="black", hatch=h, label=l)
                       for h, l in (("", "RF (left)"), ("//", "DW (middle)"), ("..", "CGLS (right)"))],
              fontsize=8, loc="upper right")
    vis = VISUAL_LABELS.get(cid)
    ax.set_title(wrap("Class shares - visual label: "
                      + (f"{vis[0]} ({vis[1]})" if vis else "not in visual sample"), 60), fontsize=9)

    # DW time series: one panel per window (no empty Apr-Sep gap); colours as in the legend below
    ts_all = [r for r in TS[cid] if r["valid_pixels"] > 0]
    shown = [k for k in ALL_CLASSES
             if any(r[f"prob_{k}"] not in ("", None) and r[f"prob_{k}"] > 0.05 for r in ts_all)]
    sub = gs[1, 2:5].subgridspec(1, 2, wspace=0.06)
    for j, (wname, wtitle) in enumerate((("febmar", "Feb-Mar 2019"), ("octdec", "Oct-Dec 2019"))):
        ax = fig.add_subplot(sub[0, j])
        ts = [r for r in ts_all if r["window"] == wname]
        dates = [datetime.datetime.strptime(r["date"], "%Y-%m-%d") for r in ts]
        for k in shown:
            ax.plot(dates, [np.nan if r[f"prob_{k}"] == "" else r[f"prob_{k}"] for r in ts],
                    marker="o", ms=3, lw=1.2, color=CLASS_COLOURS[k], label=k)
        ax.set_ylim(0, 1)
        ax.set_title(f"Dynamic World, {wtitle}\n{len(ts)} dates with valid pixels", fontsize=9)
        ax.xaxis.set_major_formatter(matplotlib.dates.DateFormatter("%d %b"))
        ax.tick_params(axis="x", labelsize=7, rotation=45)
        if j == 0:
            ax.set_ylabel("Mean DW probability in cell", labelpad=6)
        else:
            ax.tick_params(axis="y", labelleft=False)

    fig.suptitle(f"Diagnosis cell {cid}  |  mean share overlap "
                 f"{next(r['mean_share_overlap'] for r in COMP_ROWS if r['cell_id'] == cid):.3f}", fontsize=13)
    handles = [Patch(color=CLASS_COLOURS[k], label=k) for k in ALL_CLASSES]
    fig.legend(handles=handles, loc="lower center", ncol=5, fontsize=9, frameon=False,
               bbox_to_anchor=(0.5, -0.03), title="Class colours (maps, bars and time-series lines)",
               title_fontsize=9)
    path = os.path.join(OUT_DIR, f"diagnosis_{cid}.png")
    fig.savefig(path, dpi=170, bbox_inches="tight")
    DIAG_PANELS.append(path)
    plt.show()
print("Saved", DIAG_PANELS)

In [ ]:
# 8b. Sensitivity run: input window Oct-Dec only (RF retrained on 9 Oct-Dec features; DW mean over the
#     Oct-Dec images only; CGLS unchanged). Main outputs are not touched.
DW_OCT_TIF = os.path.join(OUT_DIR, "dw_mean_probabilities_octdec_only.tif")
download_geotiff(dw_mean_image(DW_COLS["octdec"]), DW_OCT_TIF, DW_TIF_BANDS)

# --- local processing
from sklearn.ensemble import RandomForestClassifier

FEATURE_BASE = ["B4", "B3", "B2", "B8", "B11", "B12", "NDVI", "NDBI", "MNDWI"]
FEATURES_MAIN = ([f + "_octdec" for f in FEATURE_BASE] + [f + "_febmar" for f in FEATURE_BASE] + ["NDVI_change"])
FEATURES_SENS = [f + "_octdec" for f in FEATURE_BASE]

def read_composite(path):
    with rasterio.open(path) as src:
        arr = src.read(masked=True).astype("float32").filled(np.nan)
        transform = src.transform
    valid = np.isfinite(arr).all(axis=0) & ~(arr == 0).all(axis=0)
    arr[:, ~valid] = np.nan
    return arr, valid, transform

def season_features(arr):
    B4, B3, B2, B8, B11, B12 = arr
    with np.errstate(divide="ignore", invalid="ignore"):
        return [B4, B3, B2, B8, B11, B12, (B8 - B4) / (B8 + B4), (B11 - B8) / (B11 + B8), (B3 - B11) / (B3 + B11)]

def feature_stack(arr_oct, arr_feb):
    a, b = season_features(arr_oct.astype("float32")), season_features(arr_feb.astype("float32"))
    return np.stack(a + b + [a[6] - b[6]]).astype("float32")      # FEATURES_MAIN order

# Same training pixels as Q1: polygons in file order (later wins), outside the grid, valid in both seasons
ta_oct, ta_v_oct, ta_tr = read_composite(q1("training_area_composite.tif"))
ta_feb, ta_v_feb, _     = read_composite(q1("training_area_composite_febmar2019.tif"))
ta_feat = feature_stack(ta_oct, ta_feb)
TA_H, TA_W = ta_oct.shape[1:]
with open(q1("training_polygons.geojson")) as f:
    polys = json.load(f)["features"]
def to_utm_coords(co):
    return list(to_utm.transform(co[0], co[1])) if isinstance(co[0], (int, float)) else [to_utm_coords(x) for x in co]
shapes = [({"type": p["geometry"]["type"], "coordinates": to_utm_coords(p["geometry"]["coordinates"])}, k)
          for k, p in enumerate(polys, start=1) if p["properties"].get("class") in CLASS_NAMES]
poly_class = {k: p["properties"]["class"] for k, p in enumerate(polys, start=1)}
praster = rasterize(shapes, out_shape=(TA_H, TA_W), transform=ta_tr, fill=0, dtype="int32")
inside = np.zeros((TA_H, TA_W), bool)
gr0, gc0 = int(round((ta_tr.f - Y1) / PIXEL_M)), int(round((X0 - ta_tr.c) / PIXEL_M))
inside[gr0:gr0 + GRID_H, gc0:gc0 + GRID_W] = True
keep = (praster > 0) & ~inside & ta_v_oct & ta_v_feb & np.isfinite(ta_feat).all(axis=0)
ri, ci = np.nonzero(keep)
y_train = np.array([CLASS_NAMES.index(poly_class[k]) for k in praster[ri, ci]])
X_main = ta_feat[:, ri, ci].T
X_sens = X_main[:, [FEATURES_MAIN.index(f) for f in FEATURES_SENS]]

RF_PARAMS = {"n_estimators": RF_TREES, "random_state": RANDOM_SEED, "class_weight": "balanced", "n_jobs": -1}
with open(q1("rf_settings.json")) as f:
    Q1_RF = json.load(f)
rf_main_check = RandomForestClassifier(**RF_PARAMS, oob_score=True).fit(X_main, y_train)
REPRO = {"training_pixels": int(len(y_train)), "q1_training_pixels": Q1_RF["cross_validation_by_patch"]["pixels"],
         "oob_score": round(float(rf_main_check.oob_score_), 4), "q1_oob_score": Q1_RF["oob_score"]}
REPRO["reproduced"] = (REPRO["training_pixels"] == REPRO["q1_training_pixels"]
                       and REPRO["oob_score"] == REPRO["q1_oob_score"])
print("Reproduction check of the main RF:", REPRO)

rf_sens = RandomForestClassifier(**RF_PARAMS, oob_score=True).fit(X_sens, y_train)

# Classify the grid with the 9 Oct-Dec features
g_oct, g_v_oct, _ = read_composite(q1("s2_composite_2019.tif"))
g_feat = np.stack(season_features(g_oct.astype("float32"))).astype("float32")
g_valid = g_v_oct & np.isfinite(g_feat).all(axis=0)
RF_SENS_MAP = np.full((GRID_H, GRID_W), 255, dtype="uint8")
RF_SENS_PROB = np.full((len(CLASS_NAMES), GRID_H, GRID_W), np.nan, dtype="float32")
proba = rf_sens.predict_proba(g_feat[:, g_valid].T)
full = np.zeros((proba.shape[0], len(CLASS_NAMES))); full[:, rf_sens.classes_] = proba
RF_SENS_MAP[g_valid] = full.argmax(axis=1)
RF_SENS_PROB[:, g_valid] = full.T
write_class_map(os.path.join(OUT_DIR, "rf_class_map_octdec_only.tif"), RF_SENS_MAP, GRID_TRANSFORM)

RF_SENS_ROWS = {}
for c in CELLS:
    rs, cs = cell_window(c)
    v = g_valid[rs, cs]
    shares = class_shares(RF_SENS_MAP[rs, cs], v)
    label, top, ts, sec, ss = label_from_shares(shares)
    mp = RF_SENS_PROB[:, rs, cs][:, v].max(axis=0) if v.any() else np.array([])
    RF_SENS_ROWS[c["cell_id"]] = {
        "cell_id": c["cell_id"], "rf_label": label, "rf_top_class": top, "rf_top_share": ts,
        "rf_second_class": sec, "rf_second_share": ss,
        "rf_mean_pixel_confidence": round(float(mp.mean()), 4) if mp.size else "",
        "rf_frac_lowconf": round(float((mp < 0.5).mean()), 4) if mp.size else "",
        "valid_pixels": int(v.sum()), **{f"share_{k}": round(shares.get(k, 0.0), 4) for k in CLASS_NAMES}}
write_csv(os.path.join(OUT_DIR, "rf_cell_labels_octdec_only.csv"), list(RF_SENS_ROWS.values()))

# Dynamic World, Oct-Dec images only (mean probability, highest class, no extra mask)
P_OCT, NOBS_OCT, MODE_OCT_RAW, DW_SENS_VALID, _ = load_dw(DW_OCT_TIF)
DW_SENS_MAP = dw_products(P_OCT, DW_SENS_VALID)
DW_SENS_MODE, DW_SENS_MODE_VALID = mode_products(MODE_OCT_RAW)
write_class_map(os.path.join(OUT_DIR, "dw_class_map_octdec_only.tif"), DW_SENS_MAP, GRID_TRANSFORM)
DW_SENS_ROWS = dw_cell_rows(DW_SENS_MAP, DW_SENS_VALID, P_OCT, NOBS_OCT, len(DW_IMAGE_INFO["octdec"]["used"]),
                            DW_SENS_MODE, DW_SENS_MODE_VALID)
write_csv(os.path.join(OUT_DIR, "dw_cell_shares_octdec_only.csv"), list(DW_SENS_ROWS.values()))

def overlap_rows(a_rows, b_rows, cid):
    return sum(min(float(a_rows[cid].get(f"share_{k}") or 0), float(b_rows[cid].get(f"share_{k}") or 0))
               for k in ALL_CLASSES)

# Two-season vs Oct-Dec-only table (all 12 cells), original columns
SENS_ROWS = []
for r in COMP_ROWS:
    cid = r["cell_id"]
    rs_, ds_ = RF_SENS_ROWS[cid], DW_SENS_ROWS[cid]
    vis = r["visual_label"]
    SENS_ROWS.append({
        "cell_id": cid, "visual_label": vis,
        "rf_octdec_only": rs_["rf_label"], "rf_two_season": r["rf_label"],
        "rf_top_share_octdec_only": rs_["rf_top_share"], "rf_top_share_two_season": r["rf_top_share"],
        "rf_same_label": same_label(rs_["rf_label"], r["rf_label"]),
        "rf_same_top_class": bool(rs_["rf_top_class"]) and rs_["rf_top_class"] == r["rf_top_class"],
        "dw_octdec_only": ds_["dw_label"], "dw_two_season": r["dw_label"],
        "dw_top_share_octdec_only": ds_["dw_top_share"], "dw_top_share_two_season": r["dw_top_share"],
        "dw_same_label": same_label(ds_["dw_label"], r["dw_label"]),
        "dw_same_top_class": bool(ds_["dw_top_class"]) and ds_["dw_top_class"] == r["dw_top_class"],
        "visual_vs_rf_octdec_only": same_label(vis, rs_["rf_label"]) if vis else "",
        "visual_vs_rf_two_season": same_label(vis, r["rf_label"]) if vis else "",
        "visual_vs_dw_octdec_only": same_label(vis, ds_["dw_label"]) if vis else "",
        "visual_vs_dw_two_season": same_label(vis, r["dw_label"]) if vis else ""})
write_csv(os.path.join(OUT_DIR, "sensitivity_two_season_vs_octdec_only.csv"), SENS_ROWS)

# Extra: before/after with "no change" wording, diagnosis flag, CGLS and mean share overlap
BA = []
for r in COMP_ROWS:
    cid = r["cell_id"]
    rs_, ds_ = RF_SENS_ROWS[cid], DW_SENS_ROWS[cid]
    ov_sens = np.mean([overlap_rows(RF_SENS_ROWS, DW_SENS_ROWS, cid), overlap_rows(RF_SENS_ROWS, CGLS_ROWS, cid),
                       overlap_rows(DW_SENS_ROWS, CGLS_ROWS, cid)])
    BA.append({"cell_id": cid, "diagnosis_cell": cid in DIAG_CELLS,
               "rf_main": r["rf_label"], "rf_octdec_only": rs_["rf_label"],
               "rf_change": "no change" if same_label(r["rf_label"], rs_["rf_label"]) else "changed",
               "dw_main": r["dw_label"], "dw_octdec_only": ds_["dw_label"],
               "dw_change": "no change" if same_label(r["dw_label"], ds_["dw_label"]) else "changed",
               "cgls": r["cgls_label"],
               "rf_vs_dw_main": r["rf_vs_dw"], "rf_vs_dw_octdec_only": same_label(rs_["rf_label"], ds_["dw_label"]),
               "rf_vs_cgls_main": r["rf_vs_cgls"], "rf_vs_cgls_octdec_only": same_label(rs_["rf_label"], r["cgls_label"]),
               "dw_vs_cgls_main": r["dw_vs_cgls"], "dw_vs_cgls_octdec_only": same_label(ds_["dw_label"], r["cgls_label"]),
               "mean_share_overlap_main": r["mean_share_overlap"],
               "mean_share_overlap_octdec_only": round(float(ov_sens), 4)})
write_csv(os.path.join(OUT_DIR, "sensitivity_before_after.csv"), BA)

SENS_SUMMARY = {
    "change": "input window two seasons (main) -> Oct-Dec only; RF retrained on the 9 Oct-Dec features with the "
              "same polygons, pixels, seed and settings; DW mean over the Oct-Dec images only; CGLS unchanged",
    "features_octdec_only": FEATURES_SENS, "oob_score_octdec_only": round(float(rf_sens.oob_score_), 4),
    "reproduction_check_main": REPRO,
    "rf_labels_changed": sum(not s["rf_same_label"] for s in SENS_ROWS),
    "dw_labels_changed": sum(not s["dw_same_label"] for s in SENS_ROWS), "n_cells": len(SENS_ROWS),
    "agreement_strict": {p: {"two_season": sum(bool(b[f"{p}_main"]) for b in BA),
                             "octdec_only": sum(bool(b[f"{p}_octdec_only"]) for b in BA), "n": len(BA)}
                         for p in ("rf_vs_dw", "rf_vs_cgls", "dw_vs_cgls")},
    "diagnosis_cells": {d: next(b for b in BA if b["cell_id"] == d) for d in DIAG_CELLS},
}
with open(os.path.join(OUT_DIR, "sensitivity_summary.json"), "w") as f:
    json.dump(SENS_SUMMARY, f, indent=1, default=str)

print(f"OOB (Oct-Dec only, 9 features): {SENS_SUMMARY['oob_score_octdec_only']}")
print(f"RF labels changed: {SENS_SUMMARY['rf_labels_changed']}/12 | DW labels changed: {SENS_SUMMARY['dw_labels_changed']}/12")
for s in SENS_ROWS:
    star = " *" if s["cell_id"] in DIAG_CELLS else "  "
    d_sh = f"{s['dw_top_share_octdec_only']:.0%}" if s["dw_top_share_octdec_only"] != "" else ""
    print(f"{s['cell_id']}{star} RF {s['rf_two_season']:<40} -> {s['rf_octdec_only']}")
    print(f"      DW {s['dw_two_season']:<40} -> {s['dw_octdec_only']} ({d_sh})")

In [ ]:
# 9. Figures: comparison_maps.png (2 x 2 pixel maps) and comparison_figure.png (3 x 4 cell-label grids)
CMAP_ALL = ListedColormap([CLASS_COLOURS[k] for k in ALL_CLASSES])
extent = [X0, X1, Y0, Y1]
comp_oct, _, _ = read_composite(q1("s2_composite_2019.tif"))
rgb = np.nan_to_num(np.clip(np.stack([comp_oct[0], comp_oct[1], comp_oct[2]], -1) / 3000.0, 0, 1))
with rasterio.open(q1("rf_class_map.tif")) as src:
    RF_MAP = src.read(1)                                  # RF indices 0-5 = first six of ALL_CLASSES
by_id = {r["cell_id"]: r for r in COMP_ROWS}

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
panels = [(None, "Sentinel-2 Oct-Dec 2019 median (B4/B3/B2, 0-3000)", None),
          (RF_MAP, "Random forest (two seasons)", "rf_label"),
          (DW_MAP, "Dynamic World (both windows pooled)", "dw_label"),
          (CG_MAP, "CGLS-LC100 2019 (100 m, nearest neighbour to 10 m)", "cgls_label")]
for ax, (arr, title, key) in zip(axes.ravel(), panels):
    if arr is None:
        ax.imshow(rgb, extent=extent, interpolation="nearest")
    else:
        ax.imshow(np.ma.masked_equal(arr, 255), extent=extent, interpolation="nearest", cmap=CMAP_ALL,
                  vmin=-0.5, vmax=len(ALL_CLASSES) - 0.5)
    for xv in np.arange(X0, X1 + 1, CELL_PX * PIXEL_M):
        ax.axvline(xv, color="white", lw=1)
    for yv in np.arange(Y0, Y1 + 1, CELL_PX * PIXEL_M):
        ax.axhline(yv, color="white", lw=1)
    for c in CELLS:
        txt = c["cell_id"] + ("*" if c["cell_id"] in VISUAL_LABELS else "")
        if key:
            txt += "\n" + by_id[c["cell_id"]][key].replace("Mixed (", "Mixed\n(")
        ax.text(c["xmin"] + 15, c["ymax"] - 15, txt, va="top", ha="left", fontsize=7,
                bbox=dict(facecolor="white", alpha=0.8, lw=0, pad=1.5))
    ax.set_xlim(X0, X1); ax.set_ylim(Y0, Y1); ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
fig.legend(handles=[Patch(color=CLASS_COLOURS[k], label=k) for k in ALL_CLASSES],
           loc="lower center", ncol=5, fontsize=9, frameon=False)
fig.text(0.01, 0.005, "* = cell in the visual-label sample", fontsize=8)
fig.tight_layout(rect=(0, 0.05, 1, 1))
MAPS_PATH = os.path.join(OUT_DIR, "comparison_maps.png")
fig.savefig(MAPS_PATH, dpi=180, bbox_inches="tight")
plt.show()

# Extra: cell labels from all four sources as 3 x 4 grids
def label_colour(label):
    s = label_set(label)
    return CLASS_COLOURS[next(iter(s))] if len(s) == 1 else "#FFFFFF"

def short(label):
    return (label.replace("Mixed (", "Mixed\n(").replace("Bare or sparse vegetation", "Bare/sparse")
                 .replace("Herbaceous vegetation", "Herbaceous").replace("Herbaceous wetland", "Herb. wetland")
                 .replace("Permanent water", "Water"))

fig, axes = plt.subplots(1, 5, figsize=(22, 4.6), gridspec_kw={"width_ratios": [1.35, 1, 1, 1, 1]})
axes[0].imshow(Image.open(q1("overview_with_grid.png")).convert("RGB"))
axes[0].set_title("Sentinel-2 Oct-Dec 2019 (B4/B3/B2, 0-3000)", fontsize=10)
axes[0].set_xticks([]); axes[0].set_yticks([])
for ax, (key, title) in zip(axes[1:], [("visual_label", "My visual labels (6 cells)"),
                                       ("rf_label", "Random forest (two seasons)"),
                                       ("dw_label", "Dynamic World (both windows pooled)"),
                                       ("cgls_label", "CGLS-LC100 2019")]):
    for r in COMP_ROWS:
        lab = r[key]
        x, y = r["col"] - 1, r["row"] - 1
        face = label_colour(lab) if lab else "#EEEEEE"
        ax.add_patch(plt.Rectangle((x, y), 1, 1, facecolor=face, edgecolor="none"))
        if lab.startswith("Mixed"):
            ax.add_patch(plt.Rectangle((x, y), 1, 1, facecolor="none", edgecolor="#BBBBBB", lw=0, hatch="//"))
        ax.add_patch(plt.Rectangle((x, y), 1, 1, facecolor="none", edgecolor="white", lw=2))
        txt = r["cell_id"] + ("\n" + short(lab) if lab else "")
        dark = face in ("#0072B2", "#555555", "#000000", "#009E73", "#D55E00")
        ax.text(x + 0.5, y + 0.5, txt, ha="center", va="center", fontsize=7,
                color="white" if dark else "black",
                bbox=None if dark or face != "#FFFFFF" else dict(facecolor="white", lw=0, pad=1))
    ax.set_xlim(0, 4); ax.set_ylim(3, 0); ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title, fontsize=10)
fig.legend(handles=[Patch(color=CLASS_COLOURS[k], label=k) for k in ALL_CLASSES]
           + [Patch(facecolor="white", edgecolor="grey", hatch="//", label="Mixed (no class > 50%)")],
           loc="lower center", ncol=5, fontsize=8, frameon=False, bbox_to_anchor=(0.55, -0.08))
fig.tight_layout()
FIG_PATH = os.path.join(OUT_DIR, "comparison_figure.png")
fig.savefig(FIG_PATH, dpi=180, bbox_inches="tight")
plt.show()
print("Saved", MAPS_PATH, "and", FIG_PATH)

In [ ]:
# 10. Comparison log, then zip the whole q1_outputs folder (Q1 outputs + comparison outputs)
comparison_log = {
    "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "inputs": {"q1_zip": Q1_ZIP, "q1_run_log_site": Q1_LOG.get("site"), "crs": EPSG,
               "aoi_utm": {"xmin": X0, "ymin": Y0, "xmax": X1, "ymax": Y1}},
    "classes": {"mine": CLASS_NAMES, "comparator_only": COMPARATOR_ONLY},
    "mixed_threshold": MIXED_THRESHOLD, "low_valid_flag": LOW_VALID_FLAG,
    "visual_labels": {k: {"label": v[0], "confidence": v[1]} for k, v in VISUAL_LABELS.items()},
    "dynamic_world": {
        "collection": DW_COLLECTION, "windows": WINDOWS,
        "matched_to_s2_scenes": DW_MATCH_S2_SCENES, "cloud_score_plus_mask": DW_APPLY_CLOUDSCORE,
        "images": {w: {"available": len(DW_IMAGE_INFO[w]["available"]), "used": len(DW_IMAGE_INFO[w]["used"]),
                       "s2_scenes_without_dw": DW_IMAGE_INFO[w]["s2_scenes_without_dw"]} for w in WINDOWS},
        "aggregation_main": "mean of each class probability over all images of both windows pooled; highest DW "
                            "class per pixel; valid = at least one observation",
        "aggregation_octdec_only": "mean of each class probability over the Oct-Dec images only",
        "aggregation_mode_version": "most frequent DW label over the same images (dw_mode_label)",
        "crosswalk": DW_CROSSWALK},
    "cgls": {"asset": CGLS_ASSET, "class_band": CGLS_CLASS_BAND, "probability_band": CGLS_PROBA_BAND,
             "native_projection": cgls_proj, "resampling": "nearest neighbour to the 10 m grid, no area weighting",
             "native_pixels": "10 m pixel centres converted to the CGLS projection CRS and floored to its pixel grid; "
                              "cgls_native_pixels = native pixels whose centre lies inside the cell; "
                              "cgls_native_pixels_touching = native pixels contributing any 10 m pixel",
             "crosswalk": {str(k): v for k, v in CGLS_CROSSWALK.items()}, "temporal_definition": "annual 2019"},
    "cell_rule": "share of valid pixels per class; one class if its share > mixed_threshold, "
                 "otherwise 'Mixed (top and second)'; 'No valid data' if no valid pixels",
    "agreement": {
        "strict": "same_label: identical single class, or both Mixed with the same two classes",
        "lenient": "a Mixed label matches either of its two classes; two Mixed labels match if they share a class",
        "share_overlap": "sum over all classes of min(share_a, share_b); a missing class counts as 0 (product pairs only)",
        "interval": "95% Wilson score interval for every count",
        "summary": SUMMARY, "summary_extended": EXT_SUMMARY},
    "diagnosis": {"rule": DIAG_RULE, "rule_selected_cells": DIAG_RULE_CELLS, "cells": DIAG_CELLS,
                  "override_reason": DIAG_OVERRIDE_REASON if DIAG_OVERRIDE else None,
                  "override_made_after_seeing_results": bool(DIAG_OVERRIDE),
                  "mean_share_overlap": {r["cell_id"]: r["mean_share_overlap"] for r in COMP_ROWS}},
    "sensitivity": SENS_SUMMARY,
    "outputs": {
        "comparison": ["dw_mean_probabilities.tif", "dw_class_map.tif", "dw_cell_shares.csv", "dw_images_used.csv",
                       "cgls_2019_on_10m_grid.tif", "cgls_class_map.tif", "cgls_cell_shares.csv",
                       "comparison_cell_labels.csv", "comparison_agreement_summary.csv", "comparison_maps.png",
                       "comparison_figure.png"],
        "extended_8c_8f": ["agreement_summary_extended.csv", "diagnosis_cells.json"]
                          + [f"dw_timeseries_{c}.csv" for c in DIAG_CELLS]
                          + [os.path.basename(p) for p in DIAG_PANELS],
        "sensitivity_8b": ["sensitivity_two_season_vs_octdec_only.csv", "sensitivity_before_after.csv",
                           "sensitivity_summary.json", "dw_mean_probabilities_octdec_only.tif",
                           "dw_class_map_octdec_only.tif", "dw_cell_shares_octdec_only.csv",
                           "rf_class_map_octdec_only.tif", "rf_cell_labels_octdec_only.csv"],
        "log": ["comparison_log.json"],
    },
    "versions": {"python": platform.python_version(), "earthengine-api": ee.__version__,
                 "rasterio": rasterio.__version__, "scikit-learn": sklearn.__version__, "numpy": np.__version__},
}
with open(os.path.join(OUT_DIR, "comparison_log.json"), "w") as f:
    json.dump(comparison_log, f, indent=1, default=str)
missing = [f for group in comparison_log["outputs"].values() for f in group
           if not os.path.exists(os.path.join(OUT_DIR, f))]
if missing:
    print("WARNING: listed outputs not found:", missing)

zip_path = shutil.make_archive(ZIP_NAME, "zip", Q1_DIR)
n_files = sum(len(fs) for _, _, fs in os.walk(Q1_DIR))
print("Wrote", zip_path, f"({os.path.getsize(zip_path)/1e6:.1f} MB) with {n_files} files")
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Not running in Colab: download", zip_path, "manually.")